In [ ]:
"""
exp13 Step 02: Parameter Neighbourhood Of E1M1 (PROTOCOL.md §4, step 02; DESCRIPTIVE: nothing is selected from it)

36 variants around the frozen E1M1: fade ratio {0.70, 0.75, 0.80, 0.85, 0.90, 0.95} x moving-average length {150, 200,
250} sessions (exit below it and original re-entry above it) x fresh-signal rule {on, off}; E1M1 = 0.85 / 200 / on. Every
variant runs as one continuous path over DEV (2015-04-17 -> 2026-04-15, exp12's candidate span; SPY and VIX cut at
2026-04-15) and over STRESS (first session with the 200-session average and vix_level_prev -> 2014-12-31; cut at
2015-03-18): 72 paths. Reproduction asserts first (frozen E1M1 +314.80% with 35 exits, E1 +108.03% with 35 exits,
buy-and-hold +235.39%), then the variant at 0.85 / 200 / on must equal the frozen E1M1 exactly.

Reported: the two grids of total return minus buy-and-hold, the share of the 36 variants beating buy-and-hold on DEV, on
STRESS and on both, and E1M1's rank on DEV. It describes the shape of the surface (a plateau or a spike); no variant is
picked or recommended. RUN_MODE: "validation_only" only (the untouched window is never read).
Outputs: store04_experiments/exp13_e1m1_robustness/step02_neighbourhood_data/. Trials: 72 (stage "robustness").
"""
None

In [ ]:
# IMPORT GENERAL PACKAGES
import pandas as pd
import numpy as np
import time
# IMPORT THE SHARED CONFIGURATION, THE EXPERIMENT CONFIGURATION AND THE PATHS
from so import config, paths
from experiments.exp13_e1m1_robustness import config as exp_config
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import write_csv_file_to_path
# IMPORT EXIT AND RE-ENTRY SIMULATION FUNCTIONS
from so.core.reentry_simulation import get_buy_and_hold_result_dict
# IMPORT THE EXPERIMENT FUNCTIONS
from experiments.exp13_e1m1_robustness.robustness import get_span_data_dict, get_dev_period_pdf, check_last_date_dict, get_stress_start_date, get_year_period_pdf, \
    get_dev_reproduction_dict, get_variant_rule_tuple, replay_rule_dict, get_path_metric_dict, check_step_trial_dict
# IMPORT TRIAL LOG FUNCTIONS
from so.core.trial_log import get_config_hash_str, log_trial_dict, read_trial_log_pdf, get_trial_count_pdf, print_previous_trial_warning

In [ ]:
# DEFINE THE RUN MODE: ONLY "validation_only" EXISTS FOR THIS EXPERIMENT (NO TEST WINDOW IS EVALUATED)
RUN_MODE_STR = "validation_only"
assert RUN_MODE_STR == "validation_only"
# CHECK THE REAL DATA ONLY OUTSIDE THE SYNTHETIC SMOKE RUN (scripts/smoke_notebook.py)
CHECK_REAL_DATA_BOOL = __name__ != "__smoke__"
# DISPLAY THE EXPERIMENT, THE PROTOCOL VERSION AND THE CONFIGURATION HASH (SHARED + EXPERIMENT CONFIG)
print(f"Run Mode:\t{RUN_MODE_STR}\tExperiment:\t{exp_config.EXPERIMENT_NAME}\tProtocol:\t{exp_config.PROTOCOL_VERSION}\tConfig Hash:\t{get_config_hash_str(exp_config)}")
# DEFINE THE OUTPUT FOLDER OF THIS STEP
output_path_str = paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step02_neighbourhood_data")
print(f"Output folder:\t{output_path_str}")
# CALL FUNCTION TO WARN ABOUT EARLIER ENTRIES OF THIS STAGE (A RERUN ADDS TRIALS)
previous_trial_count = print_previous_trial_warning(exp_config, "robustness")

In [ ]:
# CALL FUNCTION TO READ exp12's 44 REPLAY PERIODS (DATES ONLY) AND LOAD SPY AND THE VIX UP TO THEIR END (2026-04-15)
dev_period_pdf = get_dev_period_pdf(CHECK_REAL_DATA_BOOL)
dev_cutoff_date_str = str(dev_period_pdf["period_end"].iloc[-1])
dev_data_dict = get_span_data_dict(dev_cutoff_date_str)
print(f"DEV:\t{dev_period_pdf['period_start'].iloc[0]} -> {dev_cutoff_date_str}\t({len(dev_period_pdf)} periods)\tSPY sessions loaded {len(dev_data_dict['daily_pdf']):,} from {dev_data_dict['first_spy_date']}")
dev_last_date_dict = check_last_date_dict(dev_data_dict["last_date_dict"], dev_cutoff_date_str, CHECK_REAL_DATA_BOOL)
# CALL FUNCTION TO LOAD SPY AND THE VIX UP TO THE STRESS CUTOFF (2015-03-18)
stress_data_dict = get_span_data_dict(exp_config.STRESS_DATA_CUTOFF_DATE_STR)
stress_start_date = get_stress_start_date(stress_data_dict["daily_pdf"], stress_data_dict["vix_feature_pdf"])
print(f"STRESS:\t{stress_start_date} -> {exp_config.STRESS_END_DATE_STR}\tSPY sessions loaded {len(stress_data_dict['daily_pdf']):,} from {stress_data_dict['first_spy_date']}")
stress_last_date_dict = check_last_date_dict(stress_data_dict["last_date_dict"], exp_config.STRESS_DATA_CUTOFF_DATE_STR, CHECK_REAL_DATA_BOOL)

In [ ]:
"""
Reproduction (before any exp13 number): frozen E1M1, unmodified E1 and buy-and-hold over DEV (exp12's code).
"""
# CALL FUNCTION TO REPLAY AND ASSERT THE REPRODUCTION PATHS
dev_reproduction_dict = get_dev_reproduction_dict(dev_data_dict, dev_period_pdf, CHECK_REAL_DATA_BOOL)
print(f"Reproduction:\t{dev_reproduction_dict['reproduction_dict']}")
print("Reproduction asserts:\tPASSED") if CHECK_REAL_DATA_BOOL else None

In [ ]:
"""
The 36 variants over DEV and over STRESS (72 continuous paths), each against buy-and-hold over the same sessions.
"""
# DEFINE THE GRID AND THE TWO SPANS
variant_dict_list = [{"fade_ratio": fade_ratio, "ma_length": ma_length, "fresh_exit": fresh_exit_bool}
                     for ma_length in exp_config.MA_LENGTH_LIST for fresh_exit_bool in exp_config.FRESH_EXIT_BOOL_LIST for fade_ratio in exp_config.FADE_RATIO_LIST]
span_dict = {"dev": (dev_data_dict, dev_reproduction_dict["date1"], dev_reproduction_dict["date2"], dev_period_pdf),
             "stress": (stress_data_dict, stress_start_date, pd.Timestamp(exp_config.STRESS_END_DATE_STR).date(),
                        get_year_period_pdf(stress_data_dict["ohlcv_array_dict"]["session_date_list"], stress_start_date, exp_config.STRESS_END_DATE_STR))}
# LIST TO HOLD THE ROWS
start_time = time.time()
row_dict_list = []
# ITERATE OVER THE SPANS AND THE VARIANTS
for span_str, (data_dict, date1, date2, period_pdf) in span_dict.items():
    buy_hold_dict = get_buy_and_hold_result_dict(data_dict["ohlcv_array_dict"], date1, date2)
    for variant_dict in variant_dict_list:
        rule_tuple = get_variant_rule_tuple(data_dict["daily_pdf"], data_dict["vix_feature_pdf"], variant_dict["fade_ratio"], variant_dict["ma_length"], variant_dict["fresh_exit"])
        simulation_dict = replay_rule_dict(data_dict["ohlcv_array_dict"], rule_tuple, date1, date2)
        trigger_series = simulation_dict["episode_pdf"]["reentry_trigger"] if len(simulation_dict["episode_pdf"]) and "reentry_trigger" in simulation_dict["episode_pdf"].columns else pd.Series(dtype=str)
        row_dict_list.append({"span": span_str, **variant_dict, "is_e1m1": bool(variant_dict["fade_ratio"] == exp_config.FROZEN_M1_FADE_RATIO and variant_dict["ma_length"] == exp_config.FROZEN_MA_LENGTH
                                                                                and variant_dict["fresh_exit"] == exp_config.FROZEN_FRESH_EXIT_BOOL),
                              **get_path_metric_dict(simulation_dict, buy_hold_dict, period_pdf, data_dict["ohlcv_array_dict"]), "m1_reentry_count": int((trigger_series == "M1").sum())})
grid_pdf = pd.DataFrame(row_dict_list)
print(f"72 paths replayed in {time.time() - start_time:.0f} s")
# ASSERT THAT THE VARIANT AT 0.85 / 200 / ON IS THE FROZEN E1M1 (SAME TOTAL AND EXITS ON DEV)
e1m1_dev_row = grid_pdf[(grid_pdf["span"] == "dev") & grid_pdf["is_e1m1"]].iloc[0]
assert e1m1_dev_row["total_return"] == dev_reproduction_dict["frozen_simulation_dict"]["metric_dict"]["total_return"], "❌ The grid's E1M1 differs from the frozen E1M1"
assert e1m1_dev_row["exit_count"] == dev_reproduction_dict["frozen_simulation_dict"]["metric_dict"]["signal_exit_count"], "❌ The grid's E1M1 differs from the frozen E1M1"
print(f"Grid E1M1 = frozen E1M1:\t{e1m1_dev_row['total_return']:.2%} with {int(e1m1_dev_row['exit_count'])} exits")

In [ ]:
"""
The surface: total return minus buy-and-hold (percentage points) per variant, rows = average length and fresh-signal
rule, columns = fade ratio. Descriptive only.
"""
# BUILD THE TWO GRIDS
grid_table_dict = {}
for span_str in span_dict:
    span_grid_pdf = grid_pdf[grid_pdf["span"] == span_str]
    grid_table_dict[span_str] = (100 * span_grid_pdf.pivot_table(index=["ma_length", "fresh_exit"], columns="fade_ratio", values="excess_total_return")).round(2)
    print(f"\n{span_str.upper()}: total return minus buy-and-hold ({100 * span_grid_pdf['buy_hold_total_return'].iloc[0]:.2f}%), percentage points")
    display(grid_table_dict[span_str])
# COUNT THE VARIANTS BEATING BUY-AND-HOLD AND RANK E1M1 ON DEV
beat_pdf = grid_pdf.pivot_table(index=["fade_ratio", "ma_length", "fresh_exit"], columns="span", values="beats_buy_hold", aggfunc="first")
dev_grid_pdf = grid_pdf[grid_pdf["span"] == "dev"].copy()
dev_grid_pdf["dev_rank"] = dev_grid_pdf["total_return"].rank(ascending=False, method="min").astype(int)
neighbourhood_dict = {"variant_count": int(len(beat_pdf)), "dev_beat_count": int(beat_pdf["dev"].sum()), "stress_beat_count": int(beat_pdf["stress"].sum()),
                      "both_beat_count": int((beat_pdf["dev"] & beat_pdf["stress"]).sum()), "e1m1_dev_rank": int(dev_grid_pdf.loc[dev_grid_pdf["is_e1m1"], "dev_rank"].iloc[0]),
                      "e1m1_dev_total_return": float(e1m1_dev_row["total_return"]), "dev_buy_hold_total_return": float(e1m1_dev_row["buy_hold_total_return"]),
                      "stress_buy_hold_total_return": float(grid_pdf.loc[grid_pdf["span"] == "stress", "buy_hold_total_return"].iloc[0]),
                      "dev_median_excess": float(grid_pdf.loc[grid_pdf["span"] == "dev", "excess_total_return"].median()),
                      "stress_median_excess": float(grid_pdf.loc[grid_pdf["span"] == "stress", "excess_total_return"].median())}
print(f"\nVariants beating buy-and-hold:\tDEV {neighbourhood_dict['dev_beat_count']} of 36\tSTRESS {neighbourhood_dict['stress_beat_count']} of 36\t"
      f"both {neighbourhood_dict['both_beat_count']} of 36\tE1M1's rank on DEV: {neighbourhood_dict['e1m1_dev_rank']} of 36")
print(f"Median excess:\tDEV {neighbourhood_dict['dev_median_excess']:.2%}\tSTRESS {neighbourhood_dict['stress_median_excess']:.2%}")
# PREVIEW DATAFRAME
grid_pdf[["span", "fade_ratio", "ma_length", "fresh_exit", "total_return", "buy_hold_total_return", "excess_total_return", "annualized_log_excess", "log_excess_ci_low_6m",
          "log_excess_ci_high_6m", "in_market_pct", "exit_count", "m1_reentry_count", "product_s_over_r", "sharpe_ratio", "max_drawdown"]].round(4)

In [ ]:
# SAVE THE RESULTS
write_csv_file_to_path(grid_pdf, f"{output_path_str}neighbourhood_path_data.csv", "W")
write_csv_file_to_path(pd.DataFrame([neighbourhood_dict]), f"{output_path_str}neighbourhood_summary_data.csv", "W")

In [ ]:
# CHECK THE TRIAL BUDGET AND THE EXPECTED COUNT BEFORE LOGGING
trial_log_pdf = read_trial_log_pdf()
check_step_trial_dict(trial_log_pdf, "step02", len(grid_pdf))
# LOG EVERY PATH AS A ROBUSTNESS TRIAL (WINDOW COLUMNS = ITS SPAN)
logged_dict_list = []
metric_col_str_list = [col for col in grid_pdf.columns if col not in ["span", "fade_ratio", "ma_length", "fresh_exit"]]
for _, row in grid_pdf.iterrows():
    logged_dict_list.append(log_trial_dict(exp_config, "robustness", {"step": "step02", "span": row["span"], "fade_ratio": float(row["fade_ratio"]), "ma_length": int(row["ma_length"]),
                                                                      "fresh_exit": bool(row["fresh_exit"])}, {col: row[col] for col in metric_col_str_list}, False,
                                           fold_row_in={"valid_start": row["first_date"], "valid_end": row["last_date"]},
                                           note_str_in="exp13 step02 parameter neighbourhood (descriptive, nothing selected)"))
print(f"Trials logged:\t{len(logged_dict_list)}\tfirst logged_ts {logged_dict_list[0]['logged_ts']}\ttest_window_touched: {sorted({row['test_window_touched'] for row in logged_dict_list})}")

In [ ]:
# CALL FUNCTION TO READ THE TRIAL LOG
trial_log_pdf = read_trial_log_pdf()
# DISPLAY THE NUMBER OF TRIALS PER EXPERIMENT AND STAGE (MULTIPLE-TESTING RECORD)
get_trial_count_pdf(trial_log_pdf)